# ICON-D2 harvest — Gleitschirm-Insights

PLAN §5.5 track 2, phase 5. DWD publishes ICON-D2 as a rolling ~24-hour window and never as an
archive, so there is no way to fetch the wind over the Nebelhorn for a day in 2021 — and no
amount of engineering will invent one. What is possible is to start keeping it: from the first
run this notebook harvests, the project has an archive, and it grows by a day every day.

**Why this parameter set.** `tools/weather/spike_icond2.py` measured the naive harvest — 20 model
levels × 5 parameters × 49 steps — at **23.9 GB/day**. This takes the ten single-level parameters
a soaring day is actually made of, plus a coarse wind profile from five pressure levels
three-hourly over the flyable window: **181 MB/day across 250 files**. 132× cheaper, and it
answers more of the questions a pilot asks.

**Two things that are easy to get wrong and were:**

1. `hbas_sc` — *base of shallow convection* — is the cloud base. DWD does not publish `hbas_con`.
2. Where there is no convection ICON-D2 reports **0**, and averaging those in produced a cloud
   base of *113 m* on a day with cumulus at 3 080 m. Zero means absent, not ground level, so the
   mean is taken over the cells where the parameter exists and the area fraction is carried
   alongside.

Writes to the `weather` Delta table in **append** mode, keyed by model run — so a re-run of the
same run replaces rather than duplicates.

In [ ]:
%pip install -q eccodes

# eccodes ships binary wheels, so this needs no cluster-level library and no custom environment.
# The decode is the only thing Fabric's default image is missing.

In [ ]:
AOI = "oberstdorf"
# The shell bbox from config/aoi/oberstdorf.json — the whole area the app ever draws.
BBOX = {"west": 10.10, "east": 10.50, "south": 47.30, "north": 47.60}
TABLE = "weather"

BASE = "https://opendata.dwd.de/weather/nwp/icon-d2/grib"
USER_AGENT = "Gleitschirm-Insights/0.1 (open data pipeline; +https://opendata.dwd.de)"

SINGLE_LEVEL = [
    "hbas_sc", "htop_sc", "cape_ml", "cin_ml", "clct",
    "t_2m", "u_10m", "v_10m", "vmax_10m", "hzerocl",
]
# The levels DWD actually publishes. An obvious-looking 900 hPa quietly 404s.
PRESSURE_LEVELS = [1000, 975, 950, 850, 700]
PRESSURE_PARAMS = ["u", "v", "t"]

FLYABLE_STEPS = range(0, 16)
PROFILE_STEPS = range(0, 16, 3)

# Zero means "not there", not "at ground level". See the note above.
ZERO_MEANS_ABSENT = {"hbas_sc", "htop_sc"}

In [ ]:
import bz2, re, urllib.request, urllib.error
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timedelta, timezone

import numpy as np


def fetch(url, timeout=120):
    request = urllib.request.Request(url, headers={"User-Agent": USER_AGENT})
    with urllib.request.urlopen(request, timeout=timeout) as response:
        return response.read()


def latest_run():
    """The run before the newest, because the newest is usually still being written.

    DWD publishes a run's files as they are produced, so taking the newest directory lands a run
    with half its steps missing. Stepping back costs three hours of freshness and removes a whole
    class of intermittent gap.
    """
    now = datetime.now(timezone.utc)
    hour = (now.hour // 3) * 3 - 3
    if hour < 0:
        when = now.replace(hour=21, minute=0, second=0, microsecond=0) - timedelta(days=1)
    else:
        when = now.replace(hour=hour, minute=0, second=0, microsecond=0)
    return f"{when.hour:02d}", when


def grib_name(stamp, kind, param, step, level):
    stem = f"icon-d2_germany_regular-lat-lon_{kind}_{stamp}_{step:03d}"
    return f"{stem}_2d_{param}.grib2.bz2" if level is None else f"{stem}_{level}_{param}.grib2.bz2"


def read_grib(blob, parameter):
    """Decoded from memory, on one thread.

    eccodes is **not thread-safe** — decoding on six workers produced `No final 7777 in message!`
    on files that were provably intact. And an eccodes assertion aborts the process, so a malformed
    message is rejected before it is handed over rather than after.
    """
    import eccodes

    raw = bz2.decompress(blob)
    if not raw.startswith(b"GRIB") or not raw.rstrip().endswith(b"7777"):
        return None

    gid = eccodes.codes_new_from_message(raw)
    try:
        lats = eccodes.codes_get_array(gid, "latitudes")
        lons = eccodes.codes_get_array(gid, "longitudes")
        values = eccodes.codes_get_array(gid, "values")
        missing = eccodes.codes_get_double(gid, "missingValue")
    finally:
        eccodes.codes_release(gid)

    mask = (
        (lons >= BBOX["west"]) & (lons <= BBOX["east"])
        & (lats >= BBOX["south"]) & (lats <= BBOX["north"])
        & (values != missing)
    )
    inside = values[mask]
    if inside.size == 0:
        return None

    present = inside[inside > 0] if parameter in ZERO_MEANS_ABSENT else inside
    coverage = float(present.size / inside.size)
    if present.size == 0:
        return 0.0, 0.0, 0.0, int(inside.size), 0.0
    return (
        float(np.mean(present)), float(np.min(present)), float(np.max(present)),
        int(inside.size), coverage,
    )

In [ ]:
run_hour, run_when = latest_run()
stamp = f"{run_when:%Y%m%d}{run_hour}"
print(f"harvesting ICON-D2 run {stamp} over {AOI}")

jobs = []
for param in SINGLE_LEVEL:
    for step in FLYABLE_STEPS:
        jobs.append((param, step, None, f"{BASE}/{run_hour}/{param}/{grib_name(stamp, 'single-level', param, step, None)}"))
for param in PRESSURE_PARAMS:
    for level in PRESSURE_LEVELS:
        for step in PROFILE_STEPS:
            jobs.append((param, step, level, f"{BASE}/{run_hour}/{param}/{grib_name(stamp, 'pressure-level', param, step, level)}"))

print(f"{len(jobs)} files")


def download(job):
    try:
        return job, fetch(job[3])
    except (urllib.error.HTTPError, OSError) as exc:
        # A missing file is normal near the head of a run. The archive is built from what arrives;
        # a gap is recorded by absence rather than by a fabricated value.
        print(f"  · unavailable: {job[0]} step {job[1]:03d} level {job[2]} ({exc})")
        return job, None


rows, missing = [], 0
# Download in parallel — that is where the wall clock goes — and decode serially.
with ThreadPoolExecutor(max_workers=8) as pool:
    for (param, step, level, _), blob in pool.map(download, jobs):
        if blob is None:
            missing += 1
            continue
        stats = read_grib(blob, param)
        if stats is None:
            missing += 1
            continue
        mean, low, high, cells, coverage = stats
        rows.append({
            "run_ts": run_when.strftime("%Y-%m-%dT%H:00:00Z"),
            "valid_ts": (run_when + timedelta(hours=step)).strftime("%Y-%m-%dT%H:00:00Z"),
            "step_h": step,
            "parameter": param,
            "level_hpa": "" if level is None else str(level),
            "aoi": AOI,
            "mean": round(mean, 3),
            "min": round(low, 3),
            "max": round(high, 3),
            "cells": cells,
            "coverage": round(coverage, 4),
        })

print(f"{len(rows)} rows, {missing} unavailable")
assert rows, "nothing harvested — do not write an empty run over a good one"

In [ ]:
from pyspark.sql import functions as F

fresh = spark.createDataFrame(rows)
run_id = rows[0]["run_ts"]

# Idempotent by model run: a re-run replaces its own rows and leaves every other run alone. A
# scheduled job that appends blindly grows duplicates every time it is retried, and a job that
# overwrites the table throws away the archive it exists to build.
if spark.catalog.tableExists(TABLE):
    existing = spark.read.table(TABLE).where(F.col("run_ts") != run_id)
    combined = existing.unionByName(fresh, allowMissingColumns=True)
else:
    combined = fresh

combined.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(TABLE)

total = spark.read.table(TABLE)
print(f"{TABLE}: {total.count()} rows across {total.select('run_ts').distinct().count()} runs")
total.where(F.col("parameter") == "hbas_sc").where(F.col("coverage") > 0.02) \
     .select("valid_ts", "mean", "coverage").orderBy("valid_ts").show(10, truncate=False)